<a href="https://colab.research.google.com/github/Linford24/AfricaBp_Computer_Vision_for_Biodiversity_Classification/blob/main/differential_expression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update && apt-get install -y fastqc

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,190 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,616 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,314 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,848 kB]
Get:14 http://a

In [ ]:
!pip install pandas requests tqdm multiqc pydeseq2 pandas numpy matplotlib seaborn ggrepel-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.1 MB/s eta 0:00:00


In [ ]:
# Cell 1: Environment & Setup Configuration
import gzip
import logging
from pathlib import Path
import sys
import pandas as pd
import requests
from tqdm.notebook import tqdm
import gzip
import logging
import os
from pathlib import Path
import shlex
import subprocess
import sys
import psutil
import requests
from tqdm.notebook import tqdm
import logging
import os
from pathlib import Path
import subprocess
import sys
import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
# Configuration
METADATA_PATH = Path("metadata/SRP144496_metadata.tsv")
RAW_DIR = Path("raw_data")
FASTQC_DIR = Path("results/01_raw_fastqc")
MULTIQC_DIR = Path("results/02_raw_multiqc")
LOG_DIR = Path("logs")
LOG_FILE = LOG_DIR / "01_quality_control.log"

THREADS = 4
MAX_DOWNLOAD_ATTEMPTS = 3
TIMEOUT = 60

# Setup directories
for d in [RAW_DIR, FASTQC_DIR, MULTIQC_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Configure logging to both file and notebook console
logger = logging.getLogger("RNASeq_QC")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # Avoid duplicate handlers in notebook reruns

file_handler = logging.FileHandler(LOG_FILE)
stream_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter("[%(asctime)s] %(levelname)s: %(message)s")

file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info("===========================================")
logger.info("SRP144496 RNA-seq Quality Assessment")
logger.info("===========================================")

[2026-08-11 17:11:27,487] INFO: ===========================================


INFO:RNASeq_QC:===========================================


[2026-08-11 17:11:27,489] INFO: SRP144496 RNA-seq Quality Assessment


INFO:RNASeq_QC:SRP144496 RNA-seq Quality Assessment


[2026-08-11 17:11:27,493] INFO: ===========================================


INFO:RNASeq_QC:===========================================


In [ ]:
# Cell 2: Metadata Inspection & Validation
logger.info("[1/3] Validating Metadata...")

if not METADATA_PATH.exists() or METADATA_PATH.stat().st_size == 0:
    raise RuntimeError(f"Metadata file not found or empty: {METADATA_PATH}")

df_meta = pd.read_csv(METADATA_PATH, sep="\t")

required_cols = {"run_accession", "fastq_ftp", "fastq_bytes"}
missing_cols = required_cols - set(df_meta.columns)

if missing_cols:
    raise RuntimeError(f"Metadata is missing required column(s): {missing_cols}")

logger.info(
    f"Metadata valid. Loaded {len(df_meta)} samples for QC processing."
)

[2026-08-11 17:12:46,944] INFO: [1/3] Validating Metadata...


INFO:RNASeq_QC:[1/3] Validating Metadata...


[2026-08-11 17:12:46,992] INFO: Metadata valid. Loaded 16 samples for QC processing.


INFO:RNASeq_QC:Metadata valid. Loaded 16 samples for QC processing.


In [ ]:
# Cell 3: FastQ Download & Verification Helpers
def download_fastq(
    url: str,
    destination: Path,
    max_attempts: int = MAX_DOWNLOAD_ATTEMPTS,
) -> bool:
    """Download file using HTTP/HTTPS/FTP stream with progress bar."""
    if not url.startswith(("http://", "https://", "ftp://")):
        url = f"https://{url}"

    destination.unlink(missing_ok=True)

    for attempt in range(1, max_attempts + 1):
        try:
            logger.info(
                f"Download attempt {attempt}/{max_attempts} from {url}"
            )
            response = requests.get(url, stream=True, timeout=TIMEOUT)
            response.raise_for_status()

            total_size = int(response.headers.get("content-length", 0))

            with (
                open(destination, "wb") as f,
                tqdm(
                    total=total_size,
                    unit="B",
                    unit_scale=True,
                    desc=destination.name,
                    leave=False,
                ) as pbar,
            ):
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))

            return True

        except Exception as err:
            logger.warning(f"Attempt {attempt} failed: {err}")
            destination.unlink(missing_ok=True)

    return False


def validate_fastq(file_path: Path, expected_bytes: int) -> tuple[bool, str]:
    """Validate file existence, byte size, and gzip CRC integrity."""
    if not file_path.exists():
        return False, f"File not found: {file_path}"

    actual_size = file_path.stat().st_size
    if actual_size != expected_bytes:
        return (
            False,
            f"Size mismatch (expected: {expected_bytes}, actual: {actual_size})",
        )

    # Validate gzip integrity
    try:
        with gzip.open(file_path, "rb") as gz:
            while gz.read(1024 * 1024):
                pass
    except Exception as e:
        return False, f"Gzip corruption detected: {e}"

    return True, "Valid"

In [ ]:
# Cell 4: Process, Validate, and Recover All FastQ Files
logger.info("[2/3] Validating and Recovering FASTQ Files...")

recovered_count = 0
failed_count = 0

for _, row in tqdm(
    df_meta.iterrows(), total=len(df_meta), desc="Processing FASTQ samples"
):
    run_id = str(row["run_accession"])
    url = str(row["fastq_ftp"])
    expected_size = int(row["fastq_bytes"])

    fastq_file = RAW_DIR / f"{run_id}.fastq.gz"

    logger.info("-" * 50)
    logger.info(f"Sample: {run_id} | Path: {fastq_file}")

    # Initial Validation
    is_valid, msg = validate_fastq(fastq_file, expected_size)

    if is_valid:
        logger.info(f"PASS: {run_id} is intact. Skipping download.")
        continue

    logger.warning(f"FAIL: Verification failed ({msg}). Attempting recovery...")

    # Download Recovery Attempt
    if download_fastq(url, fastq_file):
        is_valid_retry, retry_msg = validate_fastq(fastq_file, expected_size)
        if is_valid_retry:
            logger.info(f"RECOVERED: {run_id} downloaded and verified.")
            recovered_count += 1
        else:
            logger.error(
                f"ERROR: Replacement file corrupt for {run_id} ({retry_msg})"
            )
            fastq_file.unlink(missing_ok=True)
            failed_count += 1
    else:
        logger.error(f"ERROR: Failed to download {run_id}")
        failed_count += 1

logger.info("=========================================================")
logger.info("FASTQ Validation Summary")
logger.info("=========================================================")
logger.info(f"Total files:    {len(df_meta)}")
logger.info(f"Recovered:      {recovered_count}")
logger.info(f"Failed:         {failed_count}")

if failed_count > 0:
    raise RuntimeError(f"{failed_count} file(s) failed verification/recovery.")

[2026-08-11 17:13:22,675] INFO: [2/3] Validating and Recovering FASTQ Files...


INFO:RNASeq_QC:[2/3] Validating and Recovering FASTQ Files...


Processing FASTQ samples:   0%|          | 0/16 [00:00<?, ?it/s]

[2026-08-11 17:13:22,725] INFO: --------------------------------------------------


INFO:RNASeq_QC:--------------------------------------------------


...

[2026-08-11 18:29:47,065] WARNING: FAIL: Verification failed (File not found: raw_data/SRR7108391.fastq.gz). Attempting recovery...


[2026-08-11 18:29:47,068] INFO: Download attempt 1/3 from https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR710/001/SRR7108391/SRR7108391.fastq.gz


INFO:RNASeq_QC:Download attempt 1/3 from https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR710/001/SRR7108391/SRR7108391.fastq.gz


SRR7108391.fastq.gz:   0%|          | 0.00/401M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# Cell 5: Run FastQC and MultiQC natively
import subprocess

logger.info("[3/3] Executing FastQC & MultiQC...")

fastq_files = list(RAW_DIR.glob("*.fastq.gz"))

# Run FastQC
logger.info(f"Running FastQC on {len(fastq_files)} files...")
fastqc_cmd = [
    "fastqc",
    "--threads",
    str(THREADS),
    "--outdir",
    str(FASTQC_DIR),
] + [str(f) for f in fastq_files]

subprocess.run(fastqc_cmd, check=True)

# Run MultiQC
logger.info("Aggregating QC reports with MultiQC...")
multiqc_cmd = [
    "multiqc",
    str(FASTQC_DIR),
    "--outdir",
    str(MULTIQC_DIR),
    "--filename",
    "SRP144496_raw_multiqc.html",
    "--force",
]

subprocess.run(multiqc_cmd, check=True)

logger.info("=================================================")
logger.info("QUALITY ASSESSMENT COMPLETED SUCCESSFULLY")
logger.info("=================================================")
logger.info(f"Report path: {MULTIQC_DIR / 'SRP144496_raw_multiqc.html'}")

[2026-08-11 18:37:02,738] INFO: [3/3] Executing FastQC & MultiQC...


INFO:RNASeq_QC:[3/3] Executing FastQC & MultiQC...


[2026-08-11 18:37:02,743] INFO: Running FastQC on 11 files...


INFO:RNASeq_QC:Running FastQC on 11 files...


[2026-08-11 18:45:57,551] INFO: Aggregating QC reports with MultiQC...


...

INFO:RNASeq_QC:QUALITY ASSESSMENT COMPLETED SUCCESSFULLY


[2026-08-11 18:46:08,218] INFO: =================================================


INFO:RNASeq_QC:=================================================


[2026-08-11 18:46:08,223] INFO: Report path: results/02_raw_multiqc/SRP144496_raw_multiqc.html


INFO:RNASeq_QC:Report path: results/02_raw_multiqc/SRP144496_raw_multiqc.html


**Alignment**

In [ ]:
# Configuration
THREADS = os.environ.get("THREADS", "8")
MIN_AVAILABLE_GB = 7
GENCODE_VERSION = "50"

RAW_DIR = Path("raw_data")
REF_DIR = Path(f"reference_genome/gencode_v{GENCODE_VERSION}")
INDEX_DIR = REF_DIR / "hisat2_index"
ALIGN_DIR = Path("results/03_alignment")
MULTIQC_DIR = Path("results/04_alignment_multiqc")
LOG_DIR = Path("logs")
LOG_FILE = LOG_DIR / "02_alignment.log"

GENOME_GZ = REF_DIR / "GRCh38.primary_assembly.genome.fa.gz"
GENOME = REF_DIR / "GRCh38.primary_assembly.genome.fa"

GTF_GZ = REF_DIR / f"gencode.v{GENCODE_VERSION}.primary_assembly.annotation.gtf.gz"
GTF = REF_DIR / f"gencode.v{GENCODE_VERSION}.primary_assembly.annotation.gtf"

SPLICE_SITES = REF_DIR / f"gencode.v{GENCODE_VERSION}.splicesites.txt"
INDEX_PREFIX = INDEX_DIR / "GRCh38"

GENOME_URL = f"https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_{GENCODE_VERSION}/GRCh38.primary_assembly.genome.fa.gz"
GTF_URL = f"https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_{GENCODE_VERSION}/gencode.v{GENCODE_VERSION}.primary_assembly.annotation.gtf.gz"

# Directory creation
for directory in [REF_DIR, INDEX_DIR, ALIGN_DIR, MULTIQC_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Logger setup
logger = logging.getLogger("RNASeq_Stage2")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_FILE)
stream_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter("[%(asctime)s] %(levelname)s: %(message)s")

file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info("===================================================")
logger.info("SRP144496 RNA-seq Alignment")
logger.info(f"HISAT2 + GRCh38 + GENCODE v{GENCODE_VERSION}")
logger.info("===================================================")

In [ ]:
# Cell 2: System Checks & Pre-flight Diagnostics
logger.info("[1/8] Checking required software...")

required_tools = [
    "hisat2",
    "hisat2-build",
    "hisat2_extract_splice_sites.py",
    "samtools",
    "multiqc",
]

missing_tools = [
    tool
    for tool in required_tools
    if subprocess.run(["which", tool], capture_output=True).returncode != 0
]
if missing_tools:
    raise RuntimeError(f"Missing required executables in PATH: {missing_tools}")

logger.info("Software checks passed.")

logger.info("[2/8] Checking available memory...")
available_gb = psutil.virtual_memory().available / (1024**3)
logger.info(f"Available RAM: approximately {available_gb:.2f} GiB")

if available_gb < MIN_AVAILABLE_GB:
    raise RuntimeError(
        f"Insufficient RAM: {available_gb:.2f} GiB available, "
        f"{MIN_AVAILABLE_GB} GiB required."
    )

logger.info("Memory check passed.")

In [ ]:
# Cell 3: Input FASTQ Integrity Checks
logger.info("[3/8] Checking FASTQ input files...")

fastq_files = sorted(list(RAW_DIR.glob("*.fastq.gz")))

if not fastq_files:
    raise RuntimeError(f"No FASTQ files found in {RAW_DIR}")

logger.info(f"Found {len(fastq_files)} FASTQ files.")


def verify_gzip(file_path: Path) -> bool:
    try:
        with gzip.open(file_path, "rb") as gz:
            while gz.read(1024 * 1024):
                pass
        return True
    except Exception:
        return False


for fastq in tqdm(fastq_files, desc="Validating FASTQ integrity"):
    logger.info(f"Checking {fastq.name}...")
    if not verify_gzip(fastq):
        raise RuntimeError(f"Corrupt FASTQ file detected: {fastq}")

logger.info("All FASTQs passed validation.")

In [ ]:
# Cell 4: Download Reference Genome and GTF Annotation
def download_stream(url: str, output_path: Path):
    """Downloads large files with a progress bar."""
    output_path.unlink(missing_ok=True)
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))

    with (
        open(output_path, "wb") as f,
        tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            desc=output_path.name,
            leave=False,
        ) as pbar,
    ):
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))


def decompress_gzip(gz_path: Path, out_path: Path):
    """Decompresses a gzip archive to disk."""
    with (
        gzip.open(gz_path, "rb") as f_in,
        open(out_path, "wb") as f_out,
        tqdm(
            total=gz_path.stat().st_size,
            unit="B",
            unit_scale=True,
            desc=f"Decompressing {gz_path.name}",
        ) as pbar,
    ):
        while chunk := f_in.read(1024 * 1024):
            f_out.write(chunk)
            pbar.update(len(chunk))


# Step 4: GRCh38 Genome Setup
logger.info("[4/8] Preparing GRCh38 Genome...")
if not GENOME.exists() or GENOME.stat().st_size == 0:
    if not GENOME_GZ.exists() or GENOME_GZ.stat().st_size == 0:
        logger.info("Downloading GRCh38 primary assembly...")
        download_stream(GENOME_URL, GENOME_GZ)

    if not verify_gzip(GENOME_GZ):
        raise RuntimeError("Genome archive is corrupted.")

    logger.info("Decompressing genome archive...")
    decompress_gzip(GENOME_GZ, GENOME)

logger.info("Genome ready.")

# Step 5: GTF Annotation & Splice Sites
logger.info("[5/8] Preparing GTF Annotation and Splice Sites...")
if not GTF.exists() or GTF.stat().st_size == 0:
    if not GTF_GZ.exists() or GTF_GZ.stat().st_size == 0:
        logger.info(f"Downloading GENCODE v{GENCODE_VERSION} annotation...")
        download_stream(GTF_URL, GTF_GZ)

    if not verify_gzip(GTF_GZ):
        raise RuntimeError("GTF annotation archive is corrupted.")

    logger.info("Decompressing GTF annotation...")
    decompress_gzip(GTF_GZ, GTF)

if not SPLICE_SITES.exists() or SPLICE_SITES.stat().st_size == 0:
    logger.info("Extracting known splice sites...")
    with open(SPLICE_SITES, "w") as out_f:
        subprocess.run(
            ["hisat2_extract_splice_sites.py", str(GTF)],
            stdout=out_f,
            check=True,
        )

logger.info("Annotation ready.")

In [ ]:
# Cell 5: HISAT2 Index Construction
logger.info("[6/8] Checking HISAT2 Index...")


def is_index_complete(prefix: Path) -> bool:
    """Checks if ht2 or ht2l index files exist."""
    ht2_exists = all(
        (prefix.parent / f"{prefix.name}.{i}.ht2").stat().st_size > 0
        if (prefix.parent / f"{prefix.name}.{i}.ht2").exists()
        else False
        for i in range(1, 9)
    )
    ht2l_exists = all(
        (prefix.parent / f"{prefix.name}.{i}.ht2l").stat().st_size > 0
        if (prefix.parent / f"{prefix.name}.{i}.ht2l").exists()
        else False
        for i in range(1, 9)
    )
    return ht2_exists or ht2l_exists


if not is_index_complete(INDEX_PREFIX):
    logger.info("Building GRCh38 HISAT2 index (this may take time)...")

    # Clean old partial indices
    for p in INDEX_DIR.glob(f"{INDEX_PREFIX.name}.*.ht2*"):
        p.unlink()

    build_cmd = [
        "hisat2-build",
        "-p",
        str(THREADS),
        str(GENOME),
        str(INDEX_PREFIX),
    ]
    subprocess.run(build_cmd, check=True)
else:
    logger.info("Complete existing HISAT2 index found. Skipping index build.")

In [ ]:
# Cell 6: Execute HISAT2 Alignment & Pipeline to Samtools
logger.info("[7/8] Aligning samples with HISAT2...")

aligned_count = 0
skipped_count = 0
failed_count = 0


def check_bam(bam_path: Path) -> bool:
    """Validates BAM integrity using samtools quickcheck."""
    if not bam_path.exists() or bam_path.stat().st_size == 0:
        return False
    res = subprocess.run(["samtools", "quickcheck", str(bam_path)])
    return res.returncode == 0


for fastq in tqdm(fastq_files, desc="Aligning Samples"):
    run_id = fastq.name.replace(".fastq.gz", "")
    sample_dir = ALIGN_DIR / run_id
    sample_dir.mkdir(parents=True, exist_ok=True)

    bam_file = sample_dir / f"{run_id}.sorted.bam"
    summary_file = sample_dir / f"{run_id}_hisat2_summary.txt"
    flagstat_file = sample_dir / f"{run_id}_flagstat.txt"

    logger.info("-" * 50)
    logger.info(f"Sample: {run_id}")

    # Resume capability check
    if check_bam(bam_file):
        logger.info("Valid BAM detected. Skipping alignment.")
        skipped_count += 1
    else:
        bam_file.unlink(missing_ok=True)
        logger.info("Running HISAT2 and piping to samtools sort...")

        hisat2_cmd = [
            "hisat2",
            "-p",
            str(THREADS),
            "-x",
            str(INDEX_PREFIX),
            "-U",
            str(fastq),
            "--known-splicesite-infile",
            str(SPLICE_SITES),
            "--summary-file",
            str(summary_file),
            "--new-summary",
        ]

        samtools_cmd = ["samtools", "sort", "-@", "2", "-o", str(bam_file), "-"]

        try:
            with open(summary_file, "a") as summary_out:
                p_hisat2 = subprocess.Popen(
                    hisat2_cmd, stdout=subprocess.PIPE, stderr=summary_out
                )
                p_samtools = subprocess.Popen(
                    samtools_cmd, stdin=p_hisat2.stdout
                )

                # Allow p_hisat2 to receive a SIGPIPE if p_samtools exits
                p_hisat2.stdout.close()
                p_samtools.communicate()

                if p_samtools.returncode != 0:
                    raise subprocess.CalledProcessError(
                        p_samtools.returncode, samtools_cmd
                    )

            if check_bam(bam_file):
                logger.info("Alignment and BAM sorting successful.")
                aligned_count += 1
            else:
                logger.error(f"BAM validation failed for {run_id}")
                bam_file.unlink(missing_ok=True)
                failed_count += 1
                continue

        except Exception as e:
            logger.error(f"Alignment pipeline failed for {run_id}: {e}")
            bam_file.unlink(missing_ok=True)
            failed_count += 1
            continue

    # BAM Indexing
    bai_file = Path(f"{bam_file}.bai")
    if not bai_file.exists() or bai_file.stat().st_size == 0:
        logger.info("Indexing BAM file...")
        subprocess.run(
            ["samtools", "index", "-@", "2", str(bam_file)], check=True
        )

    # Flagstat generation
    logger.info("Generating flagstat metrics...")
    with open(flagstat_file, "w") as flag_out:
        subprocess.run(
            ["samtools", "flagstat", "-@", "2", str(bam_file)],
            stdout=flag_out,
            check=True,
        )

logger.info("=" * 50)
logger.info("Alignment Summary:")
logger.info(f"New alignments:  {aligned_count}")
logger.info(f"Skipped:         {skipped_count}")
logger.info(f"Failed:          {failed_count}")

if failed_count > 0:
    raise RuntimeError(f"{failed_count} sample(s) failed alignment.")

In [ ]:
# Cell 7: Run MultiQC Alignment Aggregate Report
logger.info("[8/8] Running MultiQC...")

multiqc_cmd = [
    "multiqc",
    str(ALIGN_DIR),
    "--outdir",
    str(MULTIQC_DIR),
    "--filename",
    "SRP144496_alignment_multiqc_report.html",
    "--force",
]

subprocess.run(multiqc_cmd, check=True)

logger.info("==========================================================")
logger.info("SRP144496 RNA-seq Alignment completed successfully.")
logger.info("==========================================================")
logger.info(f"BAM files stored in:     {ALIGN_DIR}")
logger.info(
    f"MultiQC Report:          {MULTIQC_DIR / 'SRP144496_alignment_multiqc_report.html'}"
)
logger.info(f"Reference Genome:        {GENOME}")
logger.info(f"GTF Annotation:          {GTF}")
logger.info(f"Log File:                {LOG_FILE}")

**Quantification**

In [ ]:
# Configuration
THREADS = os.environ.get("THREADS", "6")
STRAND_MODE = "2"  # 0: unstranded, 1: stranded, 2: reversely stranded

GTF = Path(
    "reference_genome/gencode_v50/gencode.v50.primary_assembly.annotation.gtf"
)

ALIGN_DIR = Path("results/03_alignment")
COUNT_DIR = Path("results/06_quantification")
MULTIQC_DIR = Path("results/07_quantification_multiqc")
LOG_DIR = Path("logs")

COUNTS = COUNT_DIR / "SRP144496_featureCounts.txt"
MATRIX = COUNT_DIR / "SRP144496_gene_counts.tsv"
SUMMARY = Path(f"{COUNTS}.summary")
ASSIGNMENT_RATES = COUNT_DIR / "assignment_rates.tsv"
LOG_FILE = LOG_DIR / "03_quantification.log"

# Directory creation
for d in [COUNT_DIR, MULTIQC_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Logger setup
logger = logging.getLogger("RNASeq_Stage3")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_FILE)
stream_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter("[%(asctime)s] %(levelname)s: %(message)s")

file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info("====================================================")
logger.info(" SRP144496 RNA-seq Gene Quantification")
logger.info("====================================================")
logger.info(f"Strand mode: {STRAND_MODE} (reverse stranded)")

In [ ]:
# Cell 2: Software & File Diagnostics
logger.info("[1/3] Checking dependencies & input files...")

# Software check
for program in ["featureCounts", "samtools", "multiqc"]:
    if subprocess.run(["which", program], capture_output=True).returncode != 0:
        raise RuntimeError(f"Required program '{program}' not found in PATH.")

# Annotation check
if not GTF.exists() or GTF.stat().st_size == 0:
    raise RuntimeError(f"GTF annotation file not found or empty: {GTF}")

# BAM file validation
bam_files = sorted(list(ALIGN_DIR.glob("*/*.sorted.bam")))
bam_count = len(bam_files)

logger.info(f"Found {bam_count} BAM file(s).")
if bam_count != 16:
    logger.warning(
        f"Expected 16 BAM files, but found {bam_count}. Proceeding with detected files..."
    )

for bam in tqdm(bam_files, desc="Validating BAM integrity"):
    res = subprocess.run(
        ["samtools", "quickcheck", str(bam)], capture_output=True
    )
    if res.returncode != 0:
        raise RuntimeError(f"Corrupt or invalid BAM file detected: {bam}")

logger.info("All BAM files passed validation.")

In [ ]:
# Cell 3: Execute featureCounts
logger.info("[2/3] Running featureCounts...")

featurecounts_cmd = [
    "featureCounts",
    "-T",
    str(THREADS),
    "-s",
    str(STRAND_MODE),
    "-t",
    "exon",
    "-g",
    "gene_id",
    "-a",
    str(GTF),
    "-o",
    str(COUNTS),
] + [str(b) for b in bam_files]

logger.info(f"Executing featureCounts with {THREADS} threads...")
subprocess.run(featurecounts_cmd, check=True)

if not COUNTS.exists() or COUNTS.stat().st_size == 0:
    raise RuntimeError("featureCounts failed to generate output file.")

if not SUMMARY.exists() or SUMMARY.stat().st_size == 0:
    raise RuntimeError("featureCounts failed to generate summary file.")

logger.info("featureCounts completed successfully.")

In [ ]:
# Cell 4: Parse Results into Clean Count Matrix & Assignment Summaries
logger.info("[3/3] Generating DESeq2 count matrix and summary stats...")

# 1. Clean Gene Count Matrix Generation
counts_df = pd.read_csv(COUNTS, sep="\t", comment="#")

# Select Geneid (col 0) and BAM sample columns (cols 6 onward)
metadata_cols = ["Geneid", "Chr", "Start", "End", "Strand", "Length"]
sample_cols = [col for col in counts_df.columns if col not in metadata_cols]

# Clean sample headers: extract run ID from full path (e.g. results/03_alignment/SRRxxx/SRRxxx.sorted.bam -> SRRxxx)
rename_dict = {
    col: Path(col).name.replace(".sorted.bam", "") for col in sample_cols
}
clean_matrix = counts_df[["Geneid"] + sample_cols].rename(
    columns={"Geneid": "gene_id", **rename_dict}
)

clean_matrix.to_csv(MATRIX, sep="\t", index=False)
logger.info(f"Clean DESeq2 matrix saved to: {MATRIX}")

# 2. Assignment Summary Metrics
summary_df = pd.read_csv(SUMMARY, sep="\t", index_col=0)
summary_df = summary_df.rename(
    columns={col: Path(col).name.replace(".sorted.bam", "") for col in sample_cols}
)

# Calculate total and assigned reads
total_reads = summary_df.sum(axis=0)
assigned_reads = summary_df.loc["Assigned"]
assigned_percent = (assigned_reads / total_reads) * 100

rates_df = pd.DataFrame(
    {
        "total_reads": total_reads,
        "assigned_reads": assigned_reads,
        "assigned_percent": assigned_percent,
    }
)
rates_df.index.name = "sample"
rates_df.reset_index(inplace=True)

rates_df.to_csv(ASSIGNMENT_RATES, sep="\t", index=False)

# Output summary table display
print("\n" + "=" * 65)
print(f"{'SAMPLE':<15} {'TOTAL':>15} {'ASSIGNED':>15} {'ASSIGNED_%':>14}")
print("=" * 65)
for _, row in rates_df.iterrows():
    print(
        f"{row['sample']:<15} {int(row['total_reads']):>15,d} {int(row['assigned_reads']):>15,d} {row['assigned_percent']:>13.2f}%"
    )
print("=" * 65 + "\n")

# 3. MultiQC Report
logger.info("Generating MultiQC report...")
multiqc_cmd = [
    "multiqc",
    str(COUNT_DIR),
    "--outdir",
    str(MULTIQC_DIR),
    "--filename",
    "SRP144496_quantification_multiqc.html",
    "--force",
]
subprocess.run(multiqc_cmd, check=True)

logger.info("====================================================")
logger.info(" QUANTIFICATION COMPLETE")
logger.info("====================================================")
logger.info(f"Full output:          {COUNTS}")
logger.info(f"Clean Count Matrix:   {MATRIX}")
logger.info(f"Assignment Rates:     {ASSIGNMENT_RATES}")
logger.info(
    f"MultiQC Report:       {MULTIQC_DIR / 'SRP144496_quantification_multiqc.html'}"
)

**Differential Expression**

In [ ]:
#!/usr/bin/env sys.executable

# ============================================================
# Configuration
# ============================================================

COUNT_FILE = "results/06_quantification/SRP144496_gene_counts.tsv"
META_FILE = "metadata/samplesheet.csv"
GTF_FILE = "reference_genome/gencode_v50/gencode.v50.primary_assembly.annotation.gtf"
OUTDIR = "results/08_differential_expression"

# Differential-expression thresholds
PADJ_THRESHOLD = 0.05
LFC_THRESHOLD = 1.0

# Low-expression filtering
MIN_COUNT = 10
MIN_SAMPLES = 4

# Visualization parameters
TOP_HEATMAP_GENES = 50
TOP_VOLCANO_LABELS = 10

os.makedirs(OUTDIR, exist_ok=True)

# Set global plotting aesthetic
sns.set_theme(style="whitegrid")


# ============================================================
# Utility functions
# ============================================================

def stop_with_message(message: str):
    """Prints an error message and exits execution."""
    sys.exit(f"\nERROR: {message}\n")


print("\n======================================================")
print(" SRP144496 Differential Expression Analysis (Python)")
print("======================================================\n")


# ============================================================
# 1. Check required files
# ============================================================

print("[1/11] Checking required files...")

for file_path in [COUNT_FILE, META_FILE, GTF_FILE]:
    if not os.path.exists(file_path):
        stop_with_message(f"Required file not found: {file_path}")

message_files_ok = "Required files found."
print("Required files found.\n")


# ============================================================
# 2. Load count matrix
# ============================================================

print("[2/11] Loading count matrix...")

counts = pd.read_csv(COUNT_FILE, sep="\t", index_col=0)

print(f"Genes loaded:   {counts.shape[0]}")
print(f"Samples loaded: {counts.shape[1]}")

if counts.shape[1] != 16:
    stop_with_message(f"Expected 16 samples but found {counts.shape[1]}")

# Ensure integer matrix, transpose for PyDESeq2 (Samples x Genes)
try:
    counts = counts.astype(int)
except ValueError:
    stop_with_message("Non-integer values detected in the count matrix.")

if counts.isna().any().any():
    stop_with_message("NA values were detected in the count matrix.")

if (counts < 0).any().any():
    stop_with_message("Negative count values were detected.")

print("Count matrix loaded successfully.\n")


# ============================================================
# 3. Load sample metadata
# ============================================================

print("[3/11] Loading sample metadata...")

meta = pd.read_csv(META_FILE)

required_columns = ["sample", "run", "cell_line", "condition", "replicate"]
missing_columns = set(required_columns) - set(meta.columns)

if missing_columns:
    stop_with_message(f"Missing metadata columns: {', '.join(missing_columns)}")

# Match metadata order to count matrix
missing_runs = set(counts.columns) - set(meta["run"])
if missing_runs:
    stop_with_message(
        f"Count-matrix run IDs absent from metadata: {', '.join(missing_runs)}"
    )

meta = meta.set_index("run").loc[counts.columns].reset_index()
meta.index = meta["run"]

print("Metadata successfully matched to counts.\n")


# ============================================================
# 4. Load GENCODE gene annotations
# ============================================================

print("[4/11] Loading GENCODE gene annotation...")

gtf_genes = []
gene_id_regex = re.compile(r'gene_id "([^"]+)"')
gene_name_regex = re.compile(r'gene_name "([^"]+)"')
gene_type_regex = re.compile(r'gene_type "([^"]+)"')

with open(GTF_FILE, "r") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.strip().split("\t")
        if len(parts) == 9 and parts[2] == "gene":
            attr = parts[8]
            gid = gene_id_regex.search(attr)
            gname = gene_name_regex.search(attr)
            gtype = gene_type_regex.search(attr)

            gtf_genes.append(
                {
                    "gene_id": gid.group(1) if gid else np.nan,
                    "gene_symbol": gname.group(1) if gname else np.nan,
                    "gene_type": gtype.group(1) if gtype else np.nan,
                }
            )

gene_annotation = pd.DataFrame(gtf_genes).drop_duplicates(subset=["gene_id"])
gene_annotation["ensembl_gene_id"] = gene_annotation["gene_id"].str.replace(
    r"\..*$", "", regex=True
)

print(f"GENCODE genes loaded: {len(gene_annotation)}")

gene_annotation.to_csv(
    os.path.join(OUTDIR, "GENCODE_gene_annotation.tsv"),
    sep="\t",
    index=False,
)

print("GENCODE annotation loaded successfully.\n")


# ============================================================
# 5. Define experimental design
# ============================================================

print("[5/11] Constructing experimental groups...")

meta["cell_line"] = pd.Categorical(
    meta["cell_line"], categories=["HT55", "SW948"]
)
meta["condition"] = pd.Categorical(
    meta["condition"], categories=["control", "itraconazole"]
)
meta["group"] = (
    meta["cell_line"].astype(str) + "_" + meta["condition"].astype(str)
)
meta["group"] = pd.Categorical(
    meta["group"],
    categories=[
        "HT55_control",
        "HT55_itraconazole",
        "SW948_control",
        "SW948_itraconazole",
    ],
)

print("\nExperimental design:")
print(pd.crosstab(meta["cell_line"], meta["condition"]))
print("")

group_counts = meta["group"].value_counts()
if (group_counts < 2).any():
    stop_with_message(
        "One or more experimental groups have fewer than 2 replicates."
    )

meta.to_csv(os.path.join(OUTDIR, "sample_metadata_used.csv"))


# ============================================================
# 6. Construct DESeq2 dataset + filter genes
# ============================================================

print("[6/11] Constructing DESeq2 dataset...")

# PyDESeq2 expects counts as (samples x genes)
counts_pydeseq = counts.T

genes_before = counts_pydeseq.shape[1]

# Filter genes with at least 10 counts in at least 4 samples
keep = (counts_pydeseq >= MIN_COUNT).sum(axis=0) >= MIN_SAMPLES
counts_filtered = counts_pydeseq.loc[:, keep]

genes_after = counts_filtered.shape[1]

print(f"Genes before filtering: {genes_before}")
print(f"Genes after filtering:  {genes_after}")
print(f"Genes removed:          {genes_before - genes_after}\n")

pd.DataFrame(
    [
        {
            "genes_before": genes_before,
            "genes_after": genes_after,
            "genes_removed": genes_before - genes_after,
            "minimum_count": MIN_COUNT,
            "minimum_samples": MIN_SAMPLES,
        }
    ]
).to_csv(os.path.join(OUTDIR, "filtering_summary.tsv"), sep="\t", index=False)


# ============================================================
# 7. Run DESeq2
# ============================================================

print("[7/11] Running DESeq2 model...")

dds = DeseqDataSet(
    counts=counts_filtered,
    metadata=meta,
    design_factors="group",
    refit_cooks=True,
)
dds.deseq2()

print("DESeq2 model completed.\n")


# ============================================================
# 8. Export annotated normalized counts
# ============================================================

print("[8/11] Exporting normalized counts...")

# Retrieve size-factor normalized counts (Genes x Samples)
norm_counts = pd.DataFrame(
    dds.layers["normed_counts"].T,
    index=dds.var_names,
    columns=dds.obs_names,
)

normalized_output = (
    pd.DataFrame({"gene_id": norm_counts.index})
    .merge(gene_annotation, on="gene_id", how="left")
    .merge(norm_counts, left_on="gene_id", right_index=True)
)

normalized_output.to_csv(
    os.path.join(OUTDIR, "normalized_counts.tsv"), sep="\t", index=False
)

print("Normalized counts exported.\n")


# ============================================================
# 9. VST transformation + global sample QC
# ============================================================

print("[9/11] Creating sample-level QC plots...")

# PyDESeq2 handles VST transformation internally; standard log1p normalized works for QC plot scaling
vst_counts = np.log2(norm_counts + 1)

# --- PCA ---
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca_res = pca.fit_transform(vst_counts.T)
pca_df = pd.DataFrame(
    pca_res, columns=["PC1", "PC2"], index=vst_counts.columns
).join(meta)

plt.figure(figsize=(9, 7))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="condition",
    style="cell_line",
    s=100,
)

for sample, row in pca_df.iterrows():
    plt.text(row["PC1"] + 0.1, row["PC2"] + 0.1, sample, fontsize=8)

plt.xlabel(f"PC1: {round(pca.explained_variance_ratio_[0] * 100)}% variance")
plt.ylabel(f"PC2: {round(pca.explained_variance_ratio_[1] * 100)}% variance")
plt.title("SRP144496 PCA")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "PCA_all_samples.png"), dpi=300)
plt.close()

# --- Sample Correlation Heatmap ---
corr_matrix = vst_counts.corr()

plt.figure(figsize=(9, 8))
sns.clustermap(
    corr_matrix,
    cmap="vlag",
    col_colors=sns.color_palette("Set2", len(meta["condition"].unique())),
    row_colors=sns.color_palette("Set2", len(meta["condition"].unique())),
    linewidths=0,
)
plt.title("Sample correlation", pad=40)
plt.savefig(os.path.join(OUTDIR, "sample_correlation_heatmap.png"), dpi=300)
plt.close()

print("Global sample-level plots generated.\n")


# ============================================================
# 10. Differential-expression contrast function
# ============================================================


def run_contrast(cell_line: str, treated_group: str, control_group: str):
    print("\n------------------------------------------------------")
    print(f"Running contrast: {treated_group} vs {control_group}")
    print("------------------------------------------------------")

    comparison_name = f"{cell_line}_itraconazole_vs_control"
    contrast_dir = os.path.join(OUTDIR, comparison_name)
    os.makedirs(contrast_dir, exist_ok=True)

    # Run contrast hypothesis testing
    stat_res = DeseqStats(
        dds,
        contrast=["group", treated_group, control_group],
        alpha=PADJ_THRESHOLD,
    )
    stat_res.summary()
    res_df = stat_res.results_df.copy()

    res_df["gene_id"] = res_df.index
    res_df["ensembl_gene_id"] = res_df["gene_id"].str.replace(
        r"\..*$", "", regex=True
    )

    # Annotate results
    df = res_df.merge(gene_annotation, on="gene_id", how="left")

    # Classify DEGs
    df["status"] = "Not_significant"
    up_mask = (
        df["padj"].notna()
        & (df["padj"] < PADJ_THRESHOLD)
        & (df["log2FoldChange"] >= LFC_THRESHOLD)
    )
    down_mask = (
        df["padj"].notna()
        & (df["padj"] < PADJ_THRESHOLD)
        & (df["log2FoldChange"] <= -LFC_THRESHOLD)
    )

    df.loc[up_mask, "status"] = "Upregulated"
    df.loc[down_mask, "status"] = "Downregulated"

    # Export tables
    df.to_csv(
        os.path.join(contrast_dir, "all_DESeq2_results.csv"), index=False
    )

    significant = df[df["status"] != "Not_significant"]
    upregulated = df[df["status"] == "Upregulated"]
    downregulated = df[df["status"] == "Downregulated"]

    significant.to_csv(
        os.path.join(contrast_dir, "significant_DEGs.csv"), index=False
    )
    upregulated.to_csv(
        os.path.join(contrast_dir, "upregulated_DEGs.csv"), index=False
    )
    downregulated.to_csv(
        os.path.join(contrast_dir, "downregulated_DEGs.csv"), index=False
    )

    summary_table = pd.DataFrame(
        [
            {
                "Comparison": comparison_name,
                "Upregulated": len(upregulated),
                "Downregulated": len(downregulated),
                "Total_significant": len(significant),
                "padj_threshold": PADJ_THRESHOLD,
                "abs_log2FC_threshold": LFC_THRESHOLD,
            }
        ]
    )

    summary_table.to_csv(
        os.path.join(contrast_dir, "DEG_summary.tsv"), sep="\t", index=False
    )

    # --- Volcano Plot ---
    volcano = df.dropna(subset=["padj", "log2FoldChange"]).copy()
    volcano["minus_log10_padj"] = -np.log10(
        np.maximum(volcano["padj"], np.finfo(float).tiny)
    )

    volcano["plot_label"] = volcano["gene_symbol"].fillna(
        volcano["ensembl_gene_id"]
    )

    plt.figure(figsize=(8, 7))
    sns.scatterplot(
        data=volcano,
        x="log2FoldChange",
        y="minus_log10_padj",
        hue="status",
        palette={
            "Not_significant": "grey",
            "Upregulated": "red",
            "Downregulated": "blue",
        },
        alpha=0.6,
        s=15,
    )

    plt.axvline(-LFC_THRESHOLD, color="black", linestyle="--")
    plt.axvline(LFC_THRESHOLD, color="black", linestyle="--")
    plt.axhline(-np.log10(PADJ_THRESHOLD), color="black", linestyle="--")

    # Add gene labels
    top_label_genes = (
        volcano[volcano["status"] != "Not_significant"]
        .sort_values("padj")
        .head(TOP_VOLCANO_LABELS)
    )
    for _, row in top_label_genes.iterrows():
        plt.text(
            row["log2FoldChange"],
            row["minus_log10_padj"],
            row["plot_label"],
            fontsize=8,
        )

    plt.xlabel("log2 fold change")
    plt.ylabel("-log10 adjusted p-value")
    plt.title(f"{cell_line} Itraconazole vs Control")
    plt.tight_layout()
    plt.savefig(os.path.join(contrast_dir, "volcano_plot.png"), dpi=300)
    plt.close()

    # --- Heatmap ---
    if len(significant) > 0:
        top_sig = significant.sort_values("padj").head(TOP_HEATMAP_GENES)
        selected_samples = meta[meta["cell_line"] == cell_line]["run"].tolist()

        heatmap_matrix = vst_counts.loc[top_sig["gene_id"], selected_samples]

        # Row Z-score normalization
        heatmap_matrix = heatmap_matrix.sub(
            heatmap_matrix.mean(axis=1), axis=0
        ).div(heatmap_matrix.std(axis=1), axis=0)

        heatmap_labels = top_sig["gene_symbol"].fillna(
            top_sig["ensembl_gene_id"]
        )

        g = sns.clustermap(
            heatmap_matrix,
            cmap="vlag",
            yticklabels=heatmap_labels,
            xticklabels=selected_samples,
            figsize=(8, 10),
        )
        g.fig.suptitle(f"Top DEGs: {comparison_name}", y=1.02)
        plt.savefig(
            os.path.join(contrast_dir, "top50_DEG_heatmap.png"), dpi=300
        )
        plt.close()

    print(f"\nSignificant DEGs: {len(significant)}")
    print(f"  Upregulated:   {len(upregulated)}")
    print(f"  Downregulated: {len(downregulated)}")

    return summary_table


# ============================================================
# 11. Run biological comparisons
# ============================================================

print("[10/11] Running differential-expression contrasts...")

ht55_summary = run_contrast(
    cell_line="HT55",
    treated_group="HT55_itraconazole",
    control_group="HT55_control",
)

sw948_summary = run_contrast(
    cell_line="SW948",
    treated_group="SW948_itraconazole",
    control_group="SW948_control",
)

combined_summary = pd.concat([ht55_summary, sw948_summary], ignore_index=True)
combined_summary.to_csv(
    os.path.join(OUTDIR, "all_comparisons_summary.tsv"), sep="\t", index=False
)


# ============================================================
# Final report
# ============================================================

print("\n[11/11] Analysis complete.\n")
print("======================================================")
print(" DIFFERENTIAL EXPRESSION COMPLETE")
print("======================================================\n")

print(f"Genes entering DESeq2: {genes_after}\n")
print(f"Results directory:\n  {OUTDIR}\n")
print(combined_summary)
print(f"\nFinished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

In [ ]:
#!/usr/bin/env python3

"""
============================================================
SRP144496 RNA-seq
Stage 5: Shared and Cell-Line-Specific DEG Analysis

Compares:
  HT55 itraconazole vs control
  SW948 itraconazole vs control

IMPORTANT:
"HT55-only" and "SW948-only" here mean significant in one
contrast but not the other. This is descriptive and is not
equivalent to a formal interaction test.
============================================================
"""

# ============================================================
# Configuration
# ============================================================

HT55_DIR = Path("results/08_differential_expression/HT55_itraconazole_vs_control")
SW948_DIR = Path("results/08_differential_expression/SW948_itraconazole_vs_control")

HT55_ALL = HT55_DIR / "all_DESeq2_results.csv"
SW948_ALL = SW948_DIR / "all_DESeq2_results.csv"

HT55_SIG = HT55_DIR / "significant_DEGs.csv"
SW948_SIG = SW948_DIR / "significant_DEGs.csv"

OUTDIR = Path("results/09_shared_specific_DEGs")
OUTDIR.mkdir(parents=True, exist_ok=True)

PADJ_THRESHOLD = 0.05
LFC_THRESHOLD = 1.0
TOP_LABELS = 12


# ============================================================
# Utility function
# ============================================================

def stop_with_message(message: str) -> None:
    """Prints error banner and terminates execution."""
    sys.exit(f"\nERROR: {message}\n")


print("\n======================================================")
print(" Shared and Cell-Line-Specific DEG Analysis")
print("======================================================\n")


# ============================================================
# 1. Check files
# ============================================================

print("[1/8] Checking input files...")

required_files = [HT55_ALL, SW948_ALL, HT55_SIG, SW948_SIG]
missing_files = [str(f) for f in required_files if not f.exists()]

if missing_files:
    stop_with_message(f"Missing files: {', '.join(missing_files)}")

print("All required files found.\n")


# ============================================================
# 2. Load DEG results
# ============================================================

print("[2/8] Loading differential-expression results...")

ht55_all = pd.read_csv(HT55_ALL)
sw948_all = pd.read_csv(SW948_ALL)
ht55_sig = pd.read_csv(HT55_SIG)
sw948_sig = pd.read_csv(SW948_SIG)

required_columns = [
    "gene_id",
    "ensembl_gene_id",
    "gene_symbol",
    "gene_type",
    "log2FoldChange",
    "padj",
    "status"
]

for col in required_columns:
    if col not in ht55_all.columns:
        stop_with_message(f"HT55 results are missing column: {col}")
    if col not in sw948_all.columns:
        stop_with_message(f"SW948 results are missing column: {col}")

print(f"HT55 significant DEGs:  {len(ht55_sig)}")
print(f"SW948 significant DEGs: {len(sw948_sig)}\n")


# ============================================================
# 3. Determine shared and unique DEG sets
# ============================================================

print("[3/8] Identifying DEG overlap...")

ht55_genes = set(ht55_sig["gene_id"].dropna().unique())
sw948_genes = set(sw948_sig["gene_id"].dropna().unique())

shared_genes = ht55_genes.intersection(sw948_genes)
ht55_only_genes = ht55_genes - sw948_genes
sw948_only_genes = sw948_genes - ht55_genes

print(f"Shared DEGs:      {len(shared_genes)}")
print(f"HT55-only DEGs:   {len(ht55_only_genes)}")
print(f"SW948-only DEGs:  {len(sw948_only_genes)}\n")


# ============================================================
# 4. Export HT55-only and SW948-only DEG tables
# ============================================================

print("[4/8] Exporting shared and unique DEG tables...")

ht55_only = ht55_sig[ht55_sig["gene_id"].isin(ht55_only_genes)]
sw948_only = sw948_sig[sw948_sig["gene_id"].isin(sw948_only_genes)]

ht55_only.to_csv(OUTDIR / "HT55_only_DEGs.csv", index=False)
sw948_only.to_csv(OUTDIR / "SW948_only_DEGs.csv", index=False)


# ============================================================
# Construct detailed shared-DEG table
# ============================================================

shared_ht55 = ht55_sig[ht55_sig["gene_id"].isin(shared_genes)][[
    "gene_id",
    "ensembl_gene_id",
    "gene_symbol",
    "gene_type",
    "baseMean",
    "log2FoldChange",
    "pvalue",
    "padj",
    "status"
]].rename(columns={
    "baseMean": "HT55_baseMean",
    "log2FoldChange": "HT55_log2FC",
    "pvalue": "HT55_pvalue",
    "padj": "HT55_padj",
    "status": "HT55_status"
})

shared_sw948 = sw948_sig[sw948_sig["gene_id"].isin(shared_genes)][[
    "gene_id",
    "baseMean",
    "log2FoldChange",
    "pvalue",
    "padj",
    "status"
]].rename(columns={
    "baseMean": "SW948_baseMean",
    "log2FoldChange": "SW948_log2FC",
    "pvalue": "SW948_pvalue",
    "padj": "SW948_padj",
    "status": "SW948_status"
})

shared = pd.merge(shared_ht55, shared_sw948, on="gene_id", how="inner")


# ============================================================
# Classify direction of shared responses
# ============================================================

def classify_shared_direction(row):
    if row["HT55_log2FC"] > 0 and row["SW948_log2FC"] > 0:
        return "Shared_upregulated"
    elif row["HT55_log2FC"] < 0 and row["SW948_log2FC"] < 0:
        return "Shared_downregulated"
    else:
        return "Discordant_direction"

shared["shared_response"] = shared.apply(classify_shared_direction, axis=1)

# ------------------------------------------------------------
# Rank genes by significance in BOTH cell lines (pmax equivalent)
# ------------------------------------------------------------
shared["joint_padj"] = shared[["HT55_padj", "SW948_padj"]].max(axis=1)
shared = shared.sort_values(by="joint_padj", ascending=True)

shared.to_csv(OUTDIR / "shared_DEGs_detailed.csv", index=False)

# Separate biologically useful subsets
shared_up = shared[shared["shared_response"] == "Shared_upregulated"]
shared_down = shared[shared["shared_response"] == "Shared_downregulated"]
discordant = shared[shared["shared_response"] == "Discordant_direction"]

shared_up.to_csv(OUTDIR / "shared_upregulated_DEGs.csv", index=False)
shared_down.to_csv(OUTDIR / "shared_downregulated_DEGs.csv", index=False)
discordant.to_csv(OUTDIR / "shared_discordant_DEGs.csv", index=False)

print(f"Shared upregulated:   {len(shared_up)}")
print(f"Shared downregulated: {len(shared_down)}")
print(f"Discordant direction: {len(discordant)}\n")


# ============================================================
# 5. Create overall overlap summary
# ============================================================

print("[5/8] Creating overlap summaries...")

overlap_summary = pd.DataFrame({
    "Category": ["HT55_only", "Shared", "SW948_only"],
    "Genes": [len(ht55_only_genes), len(shared_genes), len(sw948_only_genes)]
})
overlap_summary.to_csv(OUTDIR / "DEG_overlap_summary.tsv", sep="\t", index=False)

direction_summary = pd.DataFrame({
    "Category": ["Shared_upregulated", "Shared_downregulated", "Discordant_direction"],
    "Genes": [len(shared_up), len(shared_down), len(discordant)]
})
direction_summary.to_csv(OUTDIR / "shared_direction_summary.tsv", sep="\t", index=False)


# ============================================================
# 6. Overlap visualizations
# ============================================================

print("[6/8] Generating overlap plots...")

# ------------------------------------------------------------
# Bar plot
# ------------------------------------------------------------
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(7, 6))

category_order = ["HT55_only", "Shared", "SW948_only"]
overlap_summary["Category"] = pd.Categorical(overlap_summary["Category"], categories=category_order, ordered=True)
overlap_summary = overlap_summary.sort_values("Category")

bars = ax.bar(overlap_summary["Category"], overlap_summary["Genes"], width=0.65, color="#34495e")

# Add count annotations above bars
for bar in bars:
    height = bar.get_height()
    ax.annotate(f"{height}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha="center", va="bottom", fontsize=10)

ax.set_ylabel("Number of significant DEGs", fontsize=12)
ax.set_title("Itraconazole-responsive DEG overlap", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(OUTDIR / "DEG_overlap_barplot.png", dpi=300)
plt.close()

# ------------------------------------------------------------
# Simple two-set Venn-style diagram (matplotlib geometric elements)
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6.4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis("off")
ax.set_title("Significant DEG overlap", fontsize=14, pad=10)

# Draw circles
circle1 = plt.Circle((4, 4), 2.5, fill=False, edgecolor="black", linewidth=1.5)
circle2 = plt.Circle((6, 4), 2.5, fill=False, edgecolor="black", linewidth=1.5)
ax.add_patch(circle1)
ax.add_patch(circle2)

# Place counts
ax.text(2.9, 4, str(len(ht55_only_genes)), fontsize=14, ha="center", va="center")
ax.text(5.0, 4, str(len(shared_genes)), fontsize=14, ha="center", va="center")
ax.text(7.1, 4, str(len(sw948_only_genes)), fontsize=14, ha="center", va="center")

# Labels
ax.text(3.0, 6.8, f"HT55\n(n = {len(ht55_genes)})", fontsize=11, ha="center", va="center")
ax.text(7.0, 6.8, f"SW948\n(n = {len(sw948_genes)})", fontsize=11, ha="center", va="center")

plt.tight_layout()
plt.savefig(OUTDIR / "DEG_overlap_venn.png", dpi=250)
plt.close()


# ============================================================
# 7. Compare log2 fold changes across ALL genes
# ============================================================

print("[7/8] Comparing treatment effect sizes...")

ht55_compare = ht55_all[[
    "gene_id", "ensembl_gene_id", "gene_symbol", "gene_type",
    "log2FoldChange", "padj", "status"
]].rename(columns={
    "log2FoldChange": "HT55_log2FC",
    "padj": "HT55_padj",
    "status": "HT55_status"
})

sw948_compare = sw948_all[[
    "gene_id", "log2FoldChange", "padj", "status"
]].rename(columns={
    "log2FoldChange": "SW948_log2FC",
    "padj": "SW948_padj",
    "status": "SW948_status"
})

comparison = pd.merge(ht55_compare, sw948_compare, on="gene_id", how="inner")

# ------------------------------------------------------------
# Classify every tested gene
# ------------------------------------------------------------
def classify_significance(row):
    ht55_sig_flag = row["HT55_status"] != "Not_significant"
    sw948_sig_flag = row["SW948_status"] != "Not_significant"

    if ht55_sig_flag and sw948_sig_flag:
        return "Significant_both"
    elif ht55_sig_flag and not sw948_sig_flag:
        return "HT55_only"
    elif not ht55_sig_flag and sw948_sig_flag:
        return "SW948_only"
    else:
        return "Neither"

comparison["significance_group"] = comparison.apply(classify_significance, axis=1)
comparison.to_csv(OUTDIR / "all_genes_HT55_SW948_comparison.csv", index=False)

# ------------------------------------------------------------
# Calculate fold-change correlation
# ------------------------------------------------------------
complete = comparison[
    np.isfinite(comparison["HT55_log2FC"]) & np.isfinite(comparison["SW948_log2FC"])
]

pearson_r = complete["HT55_log2FC"].corr(complete["SW948_log2FC"], method="pearson")
spearman_rho = complete["HT55_log2FC"].corr(complete["SW948_log2FC"], method="spearman")

correlation_summary = pd.DataFrame({
    "Metric": ["Pearson_r", "Spearman_rho"],
    "Value": [pearson_r, spearman_rho]
})
correlation_summary.to_csv(OUTDIR / "log2FC_correlation.tsv", sep="\t", index=False)

# ------------------------------------------------------------
# Select top shared genes for labelling
# ------------------------------------------------------------
label_data = shared.copy()
label_data["plot_label"] = label_data["gene_symbol"].fillna(label_data["ensembl_gene_id"])
label_data["plot_label"] = label_data["plot_label"].replace("", np.nan).fillna(label_data["ensembl_gene_id"])
label_data = label_data.head(TOP_LABELS)

# ------------------------------------------------------------
# Treatment-effect scatter plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 7))

# Gridlines & diagonal references
ax.axhline(0, color="gray", linestyle="--", alpha=0.7)
ax.axvline(0, color="gray", linestyle="--", alpha=0.7)
ax.plot([-10, 10], [-10, 10], color="gray", linestyle=":", alpha=0.7)

palette = {
    "Neither": "#bdc3c7",
    "HT55_only": "#e74c3c",
    "SW948_only": "#3498db",
    "Significant_both": "#2ecc71"
}

sns.scatterplot(
    data=comparison,
    x="HT55_log2FC",
    y="SW948_log2FC",
    hue="significance_group",
    palette=palette,
    alpha=0.45,
    s=25,
    ax=ax
)

# Label top shared genes
for _, row in label_data.iterrows():
    ax.annotate(
        row["plot_label"],
        xy=(row["HT55_log2FC"], row["SW948_log2FC"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
        alpha=0.9
    )

ax.set_xlabel("HT55 log2 fold change", fontsize=12)
ax.set_ylabel("SW948 log2 fold change", fontsize=12)
ax.set_title(f"Itraconazole treatment effects\nPearson r = {pearson_r:.3f}", fontsize=14)

# Keep axes bounds constrained to real data range
ax.set_xlim(comparison["HT55_log2FC"].min() - 0.5, comparison["HT55_log2FC"].max() + 0.5)
ax.set_ylim(comparison["SW948_log2FC"].min() - 0.5, comparison["SW948_log2FC"].max() + 0.5)

plt.tight_layout()
plt.savefig(OUTDIR / "HT55_vs_SW948_log2FC_scatter.png", dpi=300)
plt.close()


# ============================================================
# 8. Final summary
# ============================================================

print("[8/8] Analysis complete.\n")
print("======================================================")
print(" DEG OVERLAP SUMMARY")
print("======================================================\n")

print(f"HT55 significant DEGs:       {len(ht55_genes)}")
print(f"SW948 significant DEGs:      {len(sw948_genes)}")
print(f"Shared significant DEGs:     {len(shared_genes)}")
print(f"HT55-only significant DEGs:  {len(ht55_only_genes)}")
print(f"SW948-only significant DEGs: {len(sw948_only_genes)}\n")

print(f"Shared upregulated:          {len(shared_up)}")
print(f"Shared downregulated:        {len(shared_down)}")
print(f"Discordant shared DEGs:      {len(discordant)}\n")

print(f"Pearson log2FC correlation:  {pearson_r:.4f}")
print(f"Spearman correlation:        {spearman_rho:.4f}\n")

print(f"Results directory:\n  {OUTDIR.resolve()}\n")
print(f"Finished: {datetime.now().strftime('%a %b %d %H:%M:%S %Y')}\n")